# Exercise 1B Half space Gaussian shear heating diffusion


## 1. problem setting

This notebook solves one-dimensional heat diffusion in a half-space with Gaussian shear-heating near the symmetry plane. The grid is logarithmically refined near the shear zone.

**Governing equation**

$$\frac{\partial T}{\partial t}=\alpha\frac{\partial^2T}{\partial z^2}+S(z),\qquad S(z)=\frac{\tau V}{\rho c_p\sqrt{2\pi}w}\exp\!\left[-\frac12\left(\frac{z}{w}\right)^2\right],\qquad \alpha=\frac{k}{\rho c_p}.$$

**Boundary and initial conditions**

$$\frac{\partial T}{\partial z}(0,t)=0,\qquad T(L,t)=T_{far},\qquad T(z,0)=T_{far}.$$

**Parameter table**

| Symbol | Meaning | Value |
|---|---:|---:|
| $L$ | half-domain length | 1 m |
| $n_z$ | grid nodes | 101 |
| $k$ | thermal conductivity | 2.8 W m$^{-1}$ K$^{-1}$ |
| $\rho$ | density | 2700 kg m$^{-3}$ |
| $c_p$ | heat capacity | 900 J kg$^{-1}$ K$^{-1}$ |
| $T_{far}$ | far-field temperature | 293 K |
| $\tau$ | shear stress | $1.0\times10^7$ Pa |
| $V$ | slip velocity | $1.0\times10^{-5}$ m s$^{-1}$ |
| $w$ | shear-zone width | 0.02 m |
| CFL | explicit stability factor | 0.40 |
| $t_{end}$ | final time | 0.06 yr |


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from time import perf_counter
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve
from IPython.display import HTML, display

Plot_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000", "#7F7F7F", "#8B4513"]
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "axes.linewidth": 0.7,
    "axes.prop_cycle": plt.cycler(color=Plot_COLORS),
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "legend.fontsize": 6,
    "legend.frameon": False,
    "lines.linewidth": 1.1,
    "lines.markersize": 3,
    "image.cmap": "viridis",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "animation.embed_limit": 100,
})

SECONDS_PER_YR = 365.25 * 24 * 3600

CASE_ID = "Exercise1B"
ROOT = Path.cwd()
FIG = ROOT / "figures" / CASE_ID
OUT = ROOT / "outputs" / CASE_ID
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    L: float = 1                                  
    nz: int = 101
    k: float = 2.8
    rho: float = 2700.0
    cp: float = 900.0
    T_far: float = 293.0
    tau: float = 10.0e6                                     
    V: float = 1.0e-5                                                                       
    w: float = 0.02                                            
    cfl: float = 0.40
    nsave: int = 50
    t_end_yr: float = 0.06
    
    @property
    def alpha(self): 
        return self.k/(self.rho*self.cp)

cfg=Config()
                                                                                   
                                                                                             
z_min = cfg.w / 5
z = np.r_[0.0, np.geomspace(z_min, cfg.L, cfg.nz - 1)]
Q0 = cfg.tau * cfg.V / (np.sqrt(2*np.pi) * cfg.w) * np.exp(-0.5 * (z / cfg.w)**2)           
S = Q0 / (cfg.rho * cfg.cp)                                                               
S[-1] = 0.0
T0 = np.full(cfg.nz, cfg.T_far)
t_end = cfg.t_end_yr * SECONDS_PER_YR

def steady_friction_halfspace(z, H):
    I_H = np.zeros_like(H, dtype=float)
    I_H[1:] = np.cumsum(0.5 * (H[:-1] + H[1:]) * np.diff(z))
    integral_IH_from_0 = np.zeros_like(I_H, dtype=float)
    integral_IH_from_0[1:] = np.cumsum(0.5 * (I_H[:-1] + I_H[1:]) * np.diff(z))
    integral_z_to_L = integral_IH_from_0[-1] - integral_IH_from_0
    return cfg.T_far + integral_z_to_L / cfg.k

Tsteady = steady_friction_halfspace(z, Q0)
TEMP_XLIM = (0.99 * min(np.min(T0), np.min(Tsteady)), 1.01 * max(np.max(T0), np.max(Tsteady)))


## 2. shared functions


In [ ]:
def Plot_axes(ax, grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.6, pad=2)
    if grid:
        ax.grid(True, color="0.88", linewidth=0.45, alpha=0.8)
    return ax

def linf(u, ref):
    return float(np.max(np.abs(np.asarray(u, dtype=float) - np.asarray(ref, dtype=float))))

def choose_snapshot_steps(nsteps, nsave):
    return set(np.unique(np.round(np.linspace(0, nsteps, min(nsave, nsteps + 1))).astype(int)))

def plot_snapshots_depth_temperature(z_m, snapshots, times, steady, fname, title,
                                     time_scale=1.0, time_label="", xlim=None):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1, min(10, len(times)))).astype(int))
    fig, axes = plt.subplots(2, 5, figsize=(7.2, 3.5), sharex=True, sharey=True)
    panel_labels = list("abcdefghij")
    for k, ax in enumerate(axes.flat):
        if k < len(idx):
            j = idx[k]
            ax.plot(snapshots[j], z_m, color=Plot_COLORS[0], lw=1.15)
            ax.plot(steady, z_m, color="0.15", ls="--", lw=0.9)
            ax.invert_yaxis()
            if xlim is not None:
                ax.set_xlim(*xlim)
            ax.set_ylim(z_m[-1], z_m[0])
            ax.set_title(f"t={times[j] / time_scale:.3g}{time_label}", pad=2)
            Plot_axes(ax)
            ax.text(0.03, 0.94, f"({panel_labels[k]})", transform=ax.transAxes,
                    ha="left", va="top", fontsize=7, fontweight="bold")
        else:
            ax.axis("off")
    for ax in axes[-1, :]: ax.set_xlabel("T [K]")
    for ax in axes[:, 0]: ax.set_ylabel("z [m]")
    fig.suptitle(title, y=1.02, fontsize=8)
    fig.tight_layout(w_pad=0.8, h_pad=0.9)
    fig.savefig(fname)
    plt.close(fig)
    return Path(fname)

def animate_depth_temperature(z_m, snapshots, times, steady, title,
                              time_scale=1.0, time_label="", xlim=None, save_path=None):
    """Create a template-style temperature-depth animation and optionally save it as a GIF."""
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    fig, ax = plt.subplots(figsize=(3.2, 3.6))
    ax.plot(steady, z_m, "k--", lw=1.2, label="steady state")
    line, = ax.plot(snapshots[0], z_m, lw=2.0, label="numerical")
    ax.invert_yaxis()
    if xlim is None:
        xlim = TEMP_XLIM
    ax.set_xlim(*xlim)
    ax.set_ylim(z_m[-1], z_m[0])
    ax.set_xlabel("Temperature T [K]")
    ax.set_ylabel("Depth z [m]")
    ax.set_title(title)
    Plot_axes(ax)
    ax.legend(fontsize=8)
    time_text = ax.text(0.03, 0.03, "", transform=ax.transAxes)
    def update(i):
        line.set_data(snapshots[i], z_m)
        ax.set_title(f"{title}; t={times[i] / time_scale:.3g}{time_label}")
        time_text.set_text(f"t = {times[i] / time_scale:.3g}{time_label}")
        return line, time_text
    anim = FuncAnimation(fig, update, frames=len(times), interval=120, blit=True)
    if save_path is not None:
        anim.save(save_path, writer=PillowWriter(fps=5), dpi=90)
    plt.close(fig)
    return anim

def display_animation(animation):
    """Display an animation in the notebook with playback controls."""
    try:
        display(HTML(animation.to_jshtml()))
    except OSError as exc:
        print(f"Animation display skipped: {exc}")

def _png_data_uri(path):
    """Embed a saved PNG in HTML so the row display works even with absolute paths."""
    path = Path(path)
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def display_method_result_row(method_label, snapshot_path, animation):
    snapshot_uri = _png_data_uri(snapshot_path)
    anim_html = animation.to_jshtml()
    row_html = f"""
    <div style="margin: 16px 0 28px 0; width: 100%;">
      <div style="font-weight: 700; font-size: 16px; margin-bottom: 8px;">{method_label}</div>
      <div style="display: flex; flex-direction: row; gap: 18px; align-items: flex-start; width: 100%;">
        <div style="flex: 1 1 0; min-width: 0;">
          <div style="font-weight: 600; margin-bottom: 4px;">Snapshots</div>
          <img src="{snapshot_uri}" style="width: 100%; height: auto; display: block;">
        </div>
        <div style="flex: 1 1 0; min-width: 0; overflow-x: auto;">
          <div style="font-weight: 600; margin-bottom: 4px;">Animation controls</div>
          {anim_html}
        </div>
      </div>
    </div>
    """
    display(HTML(row_html))

def plot_residual_history(times, errors, fname, title, ylabel="max |T - Tsteady| [K]", time_scale=1.0, time_label=""):
    fig, ax = plt.subplots(figsize=(6, 4))
    for name, vals in errors.items():
        ax.semilogy(np.asarray(times[name]) / time_scale, np.maximum(vals, 1e-14), label=name)
    ax.set_xlabel(f"time{time_label}")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    Plot_axes(ax)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(fname)
    plt.show()

def impose_dirichlet(T):
    T=np.asarray(T,float).copy()
    T[-1]=cfg.T_far
    return T

def steady_residual_history(snapshots):
    """Residual with respect to the steady profile, measured in Kelvin."""
    return np.array([linf(s, Tsteady) for s in snapshots])

def heat_residual_summary(method, snapshots, times, elapsed, dt, nsteps):
    max_res = steady_residual_history(snapshots)
    return {
        "method": method,
        "dt_s": dt,                                                             
        "dt_yr": dt / SECONDS_PER_YR,
        "steps": int(nsteps),
        "wall_time_s": elapsed,
        "time_per_step_s": elapsed / max(nsteps, 1),
        "final_max_steady_residual_K": float(max_res[-1]),
        "max_steady_residual_history_K": max_res,
    }

def save_heat_method(method_key, method_label, snapshots, times, elapsed, dt, nsteps, store):
    method_xlim =TEMP_XLIM
    store[method_key] = heat_residual_summary(method_label, snapshots, times, elapsed, dt, nsteps)
    store[method_key]["snapshots"] = snapshots
    store[method_key]["times"] = times
    snapshot_path = plot_snapshots_depth_temperature(
        z, snapshots, times, Tsteady,
        FIG / f"{method_key}_snapshots.png",
        f"{method_label}: snapshots", time_scale=SECONDS_PER_YR, time_label=" yr",
        xlim=method_xlim,
    )
    anim = animate_depth_temperature(
        z, snapshots, times, Tsteady,
        f"{method_label}: relaxation animation", time_scale=SECONDS_PER_YR, time_label=" yr",
        xlim=method_xlim, save_path=FIG / f"{method_key}_animation.gif",
    )
    display_method_result_row(method_label, snapshot_path, anim)

fig,ax=plt.subplots(1,2,figsize=(7,3.6))
ax[0].plot(Q0,z)
ax[0].invert_yaxis()
ax[0].set(xlabel='Q [W m$^{-3}$]',ylabel='z [m]',title='Gaussian shear heating')
ax[0].grid(alpha=.3)
ax[1].plot(T0,z,'--',label='initial')
ax[1].plot(Tsteady,z,label='steady')
ax[1].invert_yaxis()
ax[1].set(xlabel='T [K]',ylabel='z [m]',title='Initial and steady')
ax[1].grid(alpha=.3)
ax[1].legend()
fig.tight_layout()
fig.savefig(FIG / "setup_source_and_steady.png")
plt.show()
results={}

skipped_methods={}

def save_skipped_method(method_key, method_label, reason, dt, nsteps, store, grid_note="production nonuniform grid"):
    store[method_key] = {
        "method": method_label,
        "status": "skipped",
        "reason": reason,
        "dt_s": dt if np.isfinite(dt) else np.nan,                                                             
        "dt_yr": dt / SECONDS_PER_YR if np.isfinite(dt) else np.nan,
        "steps": int(nsteps) if np.isfinite(nsteps) else np.nan,
        "wall_time_s": np.nan,
        "time_per_step_s": np.nan,
        "final_max_steady_residual_K": np.nan,
        "grid_note": grid_note,
    }
    print(f"{method_label} skipped: {reason}; stable dt={dt:.3e} s, steps={int(nsteps):d}")


## 3. FDM explicit


In [ ]:
dt_explicit_fdm_eval=cfg.cfl*np.min(np.diff(z))**2/cfg.alpha
nsteps_explicit_fdm_eval=int(np.ceil(t_end/dt_explicit_fdm_eval))
dt_explicit_fdm_eval=t_end/nsteps_explicit_fdm_eval
                                                                                                                    
def fem_mk(z_nodes):
    # Assemble linear-element mass and stiffness matrices.
    # Keep boundary coupling terms for the interior solve.
    n=len(z_nodes); M=np.zeros((n,n)); K=np.zeros((n,n))
    for e in range(n-1):
        h = z_nodes[e+1] - z_nodes[e]
        Me=h/6*np.array([[2,1],[1,2]],float); Ke=1/h*np.array([[1,-1],[-1,1]],float)
        sl=slice(e,e+2); M[sl,sl]+=Me; K[sl,sl]+=Ke
    return M,K

def fem_explicit_stable_dt(Mii, Kii, safety=0.45):
    # Estimate the largest stable explicit FEM step from the operator spectrum.
    eigvals=np.linalg.eigvals(np.linalg.solve(Mii,Kii))
    lambda_max=np.max(np.real(eigvals))
    return min(safety*2.0/(cfg.alpha*lambda_max), t_end)
    
M_fem_base,K_fem_base=fem_mk(z)
                                                                              
interior=slice(0,-1)
Kii_fem=K_fem_base[interior,interior]
Kbc_fem=K_fem_base[:-1,-1]*cfg.T_far

Mii_fem_consistent=M_fem_base[interior,interior]
M_fem_lumped_base=np.diag(M_fem_base.sum(axis=1))
Mii_fem_lumped=M_fem_lumped_base[interior,interior]
F_fem_consistent = (M_fem_base @ S)[interior]
F_fem_lumped = (M_fem_lumped_base @ S)[interior]

dt_explicit_fem_consistent_eval=fem_explicit_stable_dt(Mii_fem_consistent,Kii_fem)

dt_explicit_fem_lumped_eval=fem_explicit_stable_dt(Mii_fem_lumped,Kii_fem)
                                                                                                                                             
dt_common = min(dt_explicit_fdm_eval, dt_explicit_fem_consistent_eval, dt_explicit_fem_lumped_eval)
nsteps_common = int(np.ceil(t_end / dt_common))
dt_common = t_end / nsteps_common
dt_explicit_fdm = dt_common
nsteps_explicit_fdm = nsteps_common
dt_explicit_fem_consistent = dt_common                                                                                    
nsteps_explicit_fem_consistent = nsteps_common                                                                 
dt_explicit_fem_lumped = dt_common                                                                                    
nsteps_explicit_fem_lumped = nsteps_common                                                                 
print(f"Common minimum dt = {dt_common:.6e} s, steps = {nsteps_common}")
print(f"FEM explicit dt eval, consistent mass = {dt_explicit_fem_consistent_eval:.6e} s; using common dt = {dt_explicit_fem_consistent:.6e} s, steps = {nsteps_explicit_fem_consistent}")                                                      
print(f"FEM explicit dt eval, lumped mass     = {dt_explicit_fem_lumped_eval:.6e} s; using common dt = {dt_explicit_fem_lumped:.6e} s, steps = {nsteps_explicit_fem_lumped}")                                                      
EXPLICIT_STEP_LIMIT = 1000


In [ ]:
def fd_neumann_dirichlet_operator(n):
    # Build the nonuniform-grid diffusion operator for Neumann--Dirichlet boundaries.
                                                                                     
    main=np.zeros(n); upper=np.zeros(n-1); lower=np.zeros(n-1)
    h0 = z[1] - z[0]
    main[0] = -2/h0**2
    upper[0] = 2/h0**2                      
    for i in range(1, n):
        hm = z[i] - z[i-1]
        hp = z[i+1] - z[i]
        lower[i-1] = 2/(hm*(hm+hp))
        main[i] = -2/(hm*hp)
        if i < n-1:
            upper[i] = 2/(hp*(hm+hp))
    D = diags([lower,main,upper],[-1,0,1],format='csr')
    bc = np.zeros(n)
    hp = z[n] - z[n-1]
    hm = z[n-1] - z[n-2]
    bc[-1] += 2*cfg.T_far/(hp*(hm+hp))
    return D, bc

EXPLICIT_STEP_LIMIT = 1000
if nsteps_explicit_fdm > EXPLICIT_STEP_LIMIT:
    save_skipped_method(
        "fdm_explicit",
        "FDM explicit Euler",
        f"stable explicit run would require {nsteps_explicit_fdm:,} steps on the production nonuniform grid",
        dt_explicit_fdm,
        nsteps_explicit_fdm,
        skipped_methods,
    )
else:
    def fdm_explicit():
        # Build the finite-difference Laplacian on interior unknowns.
        # Advance with the explicit update using the CFL-limited time step.
        T = impose_dirichlet(T0)
        save_steps = choose_snapshot_steps(nsteps_explicit_fdm, cfg.nsave)
        snapshots = []; times = []
        n = cfg.nz - 1
        D, bc = fd_neumann_dirichlet_operator(n)
        I = eye(n, format="csr")
        A = I + dt_explicit_fdm * cfg.alpha * D
        tic = perf_counter()
        for step in range(nsteps_explicit_fdm + 1):
            if step in save_steps:
                snapshots.append(T.copy()); times.append(step * dt_explicit_fdm)
            if step == nsteps_explicit_fdm:
                break
            T[:-1] = A @ T[:-1] + dt_explicit_fdm * (S[:-1] + cfg.alpha * bc)
            T = impose_dirichlet(T)
        return np.array(snapshots), np.array(times), perf_counter() - tic, dt_explicit_fdm, nsteps_explicit_fdm

    snap, tt, elapsed, dt, ns = fdm_explicit()
    print(f"FDM explicit Euler elapsed time: {elapsed:.6f} s")
    save_heat_method("fdm_explicit", "FDM explicit Euler", snap, tt, elapsed, dt, ns, results)


## 4. FDM implicit


In [ ]:
def fdm_implicit(dt_factor=500000.0):
    # Assemble the backward-Euler system matrix for one implicit time step.
    # Solve the linear system at each time level before applying boundary values.
    dt=dt_common; nsteps=nsteps_common
    if EXPLICIT_STEP_LIMIT < nsteps_common:
        nsteps=EXPLICIT_STEP_LIMIT                                                                    
        dt=t_end/nsteps
    n=cfg.nz-1; D,bc=fd_neumann_dirichlet_operator(n)
    I = eye(n, format="csr")
    A=I-dt*cfg.alpha*D
    T=impose_dirichlet(T0); save_steps=choose_snapshot_steps(nsteps,cfg.nsave); snapshots=[]; times=[]
    tic=perf_counter()
    for step in range(nsteps+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt)
        if step==nsteps: break
        rhs=T[:-1]+dt*(S[:-1]+cfg.alpha*bc)
        T[:-1]=spsolve(A,rhs); T=impose_dirichlet(T)
    return np.array(snapshots),np.array(times),perf_counter()-tic,dt,nsteps
snap,tt,elapsed,dt,ns=fdm_implicit()
print(f'FDM backward Euler elapsed time: {elapsed:.6f} s')
save_heat_method('fdm_backward_euler','FDM backward Euler',snap,tt,elapsed,dt,ns,results)


## 5. FDM Crank--Nicolson


In [ ]:
def fdm_crank_nicolson(dt_factor=500000.0):
    # Assemble Crank--Nicolson left and right time-stepping matrices.
    # Use midpoint diffusion/wave weighting for second-order time accuracy.
    dt=dt_common; nsteps=nsteps_common
    if EXPLICIT_STEP_LIMIT < nsteps_common:
        nsteps=EXPLICIT_STEP_LIMIT                                                                    
        dt=t_end/nsteps
    n=cfg.nz-1; D,bc=fd_neumann_dirichlet_operator(n)
    I = eye(n, format="csr")
    A = I-0.5*dt*cfg.alpha*D
    B = I+0.5*dt*cfg.alpha*D
    T = impose_dirichlet(T0); save_steps=choose_snapshot_steps(nsteps,cfg.nsave); snapshots=[]; times=[]; tic=perf_counter()
    for step in range(nsteps+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt)
        if step==nsteps: break
        rhs=B@T[:-1]+dt*(S[:-1]+cfg.alpha*bc)
        T[:-1]=spsolve(A,rhs); T=impose_dirichlet(T)
    return np.array(snapshots),np.array(times),perf_counter()-tic,dt,nsteps
snap,tt,elapsed,dt,ns=fdm_crank_nicolson()
print(f'FDM Crank--Nicolson elapsed time: {elapsed:.6f} s')
save_heat_method('fdm_crank_nicolson','FDM Crank--Nicolson',snap,tt,elapsed,dt,ns,results)


## 6. Method 4 — FEM explicit, consistent and lumped mass


In [ ]:
                                                                                 
def fem_explicit_consistent_and_lumped_mass(label,lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Use the FEM stability estimate to set the explicit time step.
    if lumped:
        Mii = Mii_fem_lumped
        F = F_fem_lumped
        dt_explicit_fem = dt_explicit_fem_lumped
        nsteps_explicit_fem = nsteps_explicit_fem_lumped
        lump_diag = np.diag(Mii)
    else:
        Mii = Mii_fem_consistent
        F = F_fem_consistent
        dt_explicit_fem = dt_explicit_fem_consistent
        nsteps_explicit_fem = nsteps_explicit_fem_consistent
        lump_diag = None

    T=impose_dirichlet(T0); save_steps=choose_snapshot_steps(nsteps_explicit_fem,cfg.nsave); 
    snapshots=[]; times=[]; tic=perf_counter()
    
    for step in range(nsteps_explicit_fem+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt_explicit_fem)
        if step==nsteps_explicit_fem: break
        u = T[:-1]
        rhs=Mii@u + dt_explicit_fem*(F - cfg.alpha*(Kii_fem@u + Kbc_fem))
        T[:-1] = np.linalg.solve(Mii,rhs)
        T=impose_dirichlet(T)
    return np.array(snapshots),np.array(times),perf_counter()-tic,dt_explicit_fem,nsteps_explicit_fem
for key,label,lumped in [
    ('fem_explicit_consistent','FEM explicit Euler, consistent mass',False),
    ('fem_explicit_lumped','FEM explicit Euler, lumped mass',True)
]:
    dt_exp = dt_explicit_fem_lumped if lumped else dt_explicit_fem_consistent                                           
    ns_exp = nsteps_explicit_fem_lumped if lumped else nsteps_explicit_fem_consistent                                              
    if ns_exp > EXPLICIT_STEP_LIMIT:
        save_skipped_method(
            key, label,
            f"stable explicit run would require {ns_exp:,} steps on the production nonuniform grid",
            dt_exp, ns_exp, skipped_methods,
        )
        continue
    snap,tt,elapsed,dt,ns=fem_explicit_consistent_and_lumped_mass(label,lumped)
    print(f'{label} elapsed time: {elapsed:.6f} s')
    save_heat_method(key,label,snap,tt,elapsed,dt,ns,results)


## 7. Method 5 — FEM backward, consistent and lumped mass


In [ ]:
def fem_backward_consistent_and_lumped_mass(label,lumped=False,dt_factor=500000.0):
    # Select either the consistent or lumped FEM mass matrix.
    # Assemble the backward FEM matrix or block system for implicit stepping.
    if lumped:
        Mii = Mii_fem_lumped
        F = F_fem_lumped
        dt_ref_fem = dt_explicit_fem_lumped
        lump_diag = np.diag(Mii)
    else:
        Mii = Mii_fem_consistent
        F = F_fem_consistent
        dt_ref_fem = dt_explicit_fem_consistent
        lump_diag = None
        
    dt_fem=dt_common; nsteps_fem=nsteps_common                                                                                 
    if EXPLICIT_STEP_LIMIT < nsteps_common:
        nsteps_fem=EXPLICIT_STEP_LIMIT                                                                    
        dt_fem=t_end/nsteps_fem
    A = Mii + dt_fem * cfg.alpha * Kii_fem
    T = impose_dirichlet(T0); save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    snapshots = []; times = []; tic = perf_counter()
    for step in range(nsteps_fem+1):
        if step in save_steps: snapshots.append(T.copy()); times.append(step*dt_fem)
        if step==nsteps_fem: break
        rhs = Mii @ T[:-1] + dt_fem * (F - cfg.alpha * Kbc_fem)
        T[:-1]=np.linalg.solve(A,rhs)
        T=impose_dirichlet(T)
    return np.array(snapshots),np.array(times),perf_counter()-tic,dt_fem,nsteps_fem
    
for key,label,lumped in [
    ('fem_be_consistent','FEM backward Euler, consistent mass',False),
    ('fem_be_lumped','FEM backward Euler, lumped mass',True)
]:
    snap,tt,elapsed,dt,ns=fem_backward_consistent_and_lumped_mass(label,lumped)
    print(f'{label} elapsed time: {elapsed:.6f} s')
    save_heat_method(key,label,snap,tt,elapsed,dt,ns,results)


## 8. Method 6 — pseudospectral cosine method for heat or wave equation


In [ ]:
def pseudospectral_cosine_method_for_heat_or_wave_equation(nmodes=None):
    # Transform the initial field and source into modal coefficients.
    # Advance spectral modes independently in time before reconstructing the field.
    """Neumann at z=0 and Dirichlet at z=L reqinteriorre cos(k_m z), k_m=(m+1/2)pi/L."""
    theta0=impose_dirichlet(T0)-Tsteady
    theta0_int = theta0[0:-1]
    z_unknown = z[0:-1]
    n_int = len(z_unknown); modes = np.arange(n_int)
    k = (modes + 0.5) * np.pi / cfg.L
    lam = ((modes + 0.5) * np.pi / cfg.L) ** 2
                                                                  
    coeff = np.zeros(n_int)
    for j, kj in enumerate(k):
        coeff[j] = (2.0 / cfg.L) * np.trapezoid(theta0 * np.cos(kj * z), x=z)
    Phi = np.cos(np.outer(z_unknown, k))
    nsteps=nsteps_common; dt=dt_common                                                                            
    if EXPLICIT_STEP_LIMIT < nsteps_common:
        nsteps=EXPLICIT_STEP_LIMIT                                                                                                                                                
        dt=t_end/nsteps
    save_steps=choose_snapshot_steps(nsteps,cfg.nsave)
    amp = np.exp(-cfg.alpha * lam * dt)                                                           
    coeff_t = coeff.copy()                                       
    T = Tsteady.copy(); T[0:-1] += Phi @ coeff_t                                                              
    snapshots=[]; times=[]
    tic=perf_counter()
    for step in range(nsteps+1):
        if step in save_steps:   
            snapshots.append(impose_dirichlet(T))
            times.append(step*dt)
        if step==nsteps: break
        coeff_t = amp * coeff_t                                                      
        T = Tsteady.copy()
        T[0:-1] += Phi @ coeff_t
    return np.array(snapshots),np.array(times),perf_counter()-tic,dt,nsteps
    
snap,tt,elapsed,dt,ns=pseudospectral_cosine_method_for_heat_or_wave_equation()
print(f'Pseudospectral cosine ND elapsed time: {elapsed:.6f} s')
save_heat_method('pseudospectral_cosine_nd','Pseudospectral cosine Neumann--Dirichlet relaxation',snap,tt,elapsed,dt,ns,results)


## 9. Final residual


In [ ]:
                                                                         
summary = pd.DataFrame([
    {k: v for k, v in d.items() if not isinstance(v, np.ndarray) and k not in ["snapshots", "times"]}
    for d in results.values()
]).sort_values("final_max_steady_residual_K")

display(summary)
summary.to_csv(OUT / "exercise1b_residual_efficiency_summary.csv", index=False)

                                                                              
HISTORY_FIGSIZE = (6, 4)
COMBINED_FIGSIZE = (6, 4)
PANEL_BASE_FIGSIZE = (4.4, 4.8)
PANEL_NCOLS = 3

                                         
                                             
residual_hist = {v["method"]: v["max_steady_residual_history_K"] for v in results.values()}
time_hist = {v["method"]: v["times"] for v in results.values()}
fig, ax = plt.subplots(figsize=HISTORY_FIGSIZE)
for name, vals in residual_hist.items():
    ax.semilogy(time_hist[name] / SECONDS_PER_YR, np.maximum(vals, 1e-14), label=name)
ax.set_xlabel("time [yr]")
ax.set_ylabel("max |T - Tsteady| [K]")
ax.set_title("Exercise 1B: convergence to steady state")
Plot_axes(ax)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "exercise1b_max_steady_residual_history.png")
plt.show()

                                                                               
fig, ax = plt.subplots(figsize=COMBINED_FIGSIZE)
for key, v in results.items():
    ax.plot(v["snapshots"][-1] - Tsteady, z, label=v["method"])
ax.axvline(0.0, color="k", lw=0.7)
ax.invert_yaxis()
ax.set_ylim(z[-1], z[0])
ax.set_xlabel("T - Tsteady [K]")
ax.set_ylabel("distance from shear-zone centre z [m]")
ax.set_title("Final steady residual profile")
Plot_axes(ax)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIG / "exercise1b_final_residual_profiles_combined.png")
plt.show()

                                                                                                    
n_methods = len(results)
ncols = PANEL_NCOLS
nrows = int(np.ceil(n_methods / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(PANEL_BASE_FIGSIZE[0] * ncols, PANEL_BASE_FIGSIZE[1] * nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, (key, v) in zip(axes, results.items()):
    residual = v["snapshots"][-1] - Tsteady
    ax.plot(residual, z)
    ax.axvline(0.0, color="k", lw=0.7)
    ax.invert_yaxis()
    ax.set_ylim(z[-1], z[0])
    ax.set_title(v["method"], fontsize=9)
    ax.set_xlabel("T - Tsteady [K]")
    Plot_axes(ax)
for ax in axes[::ncols]:
    ax.set_ylabel("z [m]")
for ax in axes[n_methods:]:
    ax.axis("off")
fig.suptitle("Exercise 1B: final residual profile by method", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "exercise1b_final_residual_profiles_by_method.png")
plt.show()

print("Summary CSV saved to:", OUT / "exercise1b_residual_efficiency_summary.csv")
print("Figures saved in:", FIG)


## 10. L2 error


In [ ]:
def green_ND_heat_reference(times, nmodes=None, nz_green=None):
    """
    Explicit Green-function reference for Exercise 1B.

    PDE for perturbation:
        theta_t = alpha theta_zz
        theta_z(0,t) = 0
        theta(L,t) = 0

    Green basis:
        cos((n+1/2)*pi*z/L), n = 0,1,2,...

    T(z,t) = Tsteady(z) + sum_n a_n cos(k_n z)
             exp[-alpha*k_n^2*t]

    """
    times = np.asarray(times, dtype=float)

    if nmodes is None:
        nmodes = max(500, 8 * (cfg.nz - 1))

    if nz_green is None:
        nz_green = max(2000, 20 * cfg.nz)                                                                

    z_green = np.linspace(0.0, cfg.L, nz_green)                              

    T0_green = np.interp(z_green, z, impose_dirichlet(T0))                                                                        
    Tsteady_green = np.interp(z_green, z, Tsteady)                                                                      
    theta0_green = T0_green - Tsteady_green                                              

    n = np.arange(0, nmodes)
    k = (n + 0.5) * np.pi / cfg.L

    Phi_green = np.cos(np.outer(z_green, k))                                       

                                                            
                                                                              
    coeff = (2.0 / cfg.L) * np.trapezoid(theta0_green[:, None] * Phi_green, x=z_green, axis=0)                                         

    tic = perf_counter()
    snapshots = []

    for t in times:
        theta_green = Phi_green @ (coeff * np.exp(-cfg.alpha * k**2 * t))                                                   
        T_green = Tsteady_green + theta_green                                               

                                                    
                                                           
        T_green[-1] = cfg.T_far

        T_on_num = np.interp(z, z_green, T_green)                                                                                    
        T_on_num[-1] = cfg.T_far                                                 

        snapshots.append(T_on_num)

    elapsed = perf_counter() - tic
    return np.asarray(snapshots), elapsed

rows = []

for key, v in results.items():
    times = np.asarray(v["times"], dtype=float)
    T_num = np.asarray(v["snapshots"], dtype=float)

    T_ref, ref_elapsed = green_ND_heat_reference(times)

    diff = T_num - T_ref

    l2_t = np.sqrt(np.trapezoid(diff**2, x=z, axis=1))
    ref_l2_t = np.sqrt(np.trapezoid(T_ref**2, x=z, axis=1))
    rel_l2_t = l2_t / np.maximum(ref_l2_t, 1e-300)

    v["green_reference_snapshots"] = T_ref
    v["L2_error_vs_Green"] = l2_t
    v["relative_L2_error_vs_Green"] = rel_l2_t
    v["Linf_error_vs_Green"] = np.max(np.abs(diff), axis=1)

    rows.append({
        "method": v["method"],
        "dt": v.get("dt_s", np.nan),                                                    
        "steps": v.get("steps", np.nan),
        "wall time": v.get("wall_time_s", np.nan),
        "final_L2_error_vs_Green": float(l2_t[-1]),
        "relative L2": float(np.max(rel_l2_t)),                                                            
        "final relative L2": float(rel_l2_t[-1]),
        "max_L2_error_vs_Green": float(np.max(l2_t)),
        "max_Linf_error_vs_Green": float(np.max(v["Linf_error_vs_Green"])),
    })


green_l2_summary = pd.DataFrame(rows)[["method", "dt", "steps", "wall time", "relative L2", "max_L2_error_vs_Green", "final relative L2"]].sort_values(
    "relative L2"
)

display(green_l2_summary)

green_l2_summary.to_csv(
    OUT / "exercise1b_l2_error_vs_green_ND.csv",
    index=False
)


In [ ]:

def green_ND_heat_reference_linear(times, nmodes=None, nz_green=None):
    """
    """
    times = np.asarray(times, dtype=float)

    if nmodes is None:
        nmodes = max(500, 8 * (cfg.nz - 1))

    if nz_green is None:
        nz_green = max(2000, 20 * cfg.nz)

    z_green = np.linspace(0.0, cfg.L, nz_green)                              

    T0_green = np.interp(z_green, z, impose_dirichlet(T0))                                                         
    Tsteady_green = np.interp(z_green, z, Tsteady)                                                            
    theta0_green = T0_green - Tsteady_green

    n = np.arange(0, nmodes)
    k = (n + 0.5) * np.pi / cfg.L
    Phi_green = np.cos(np.outer(z_green, k))

    coeff = (2.0 / cfg.L) * np.trapezoid(
        theta0_green[:, None] * Phi_green,
        x=z_green,
        axis=0
    )                                         

    snapshots = []
    tic = perf_counter()

    for t in times:
        theta_green = Phi_green @ (coeff * np.exp(-cfg.alpha * k**2 * t))
        T_green = Tsteady_green + theta_green
        T_green[-1] = cfg.T_far

        T_on_num = np.interp(z, z_green, T_green)                                                               
        T_on_num[-1] = cfg.T_far

        snapshots.append(T_on_num)

    return np.asarray(snapshots), perf_counter() - tic


                                                                                        
nsteps_green = nsteps_common
dt_green = dt_common
save_steps_green = choose_snapshot_steps(nsteps_green, cfg.nsave)
save_steps_green = sorted(list(save_steps_green))                                                                   
tt_green = np.array(save_steps_green, dtype=float) * dt_green

snap_green, elapsed_green = green_ND_heat_reference_linear(tt_green)

green_results = {}                                                                            
save_heat_method(
    "green_nd_linear_reference",
    "Green ND reference, linear interpolation",
    snap_green,
    tt_green,
    elapsed_green,
    dt_green,
    nsteps_green,
    green_results,
)

print(f"Green ND reference elapsed time: {elapsed_green:.6f} s")
